# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aiman-bscs5/flyrank_ml_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Lane 4: CTR / Engagement Opportunity Scoring, and I'd call this ranking/scoring, not classification. I'm not trying to predict if a page will decline (that's already `is_declining_label`/`trend_direction`, a different lane's target) — I'm trying to answer "which pages should a reviewer fix first," and that needs an ordered list, not a yes/no flag. It's not clustering either since I already know what I'm looking for: pages under-clicking for how well they rank. So every eligible page gets a priority score instead of a label, and reviewers just work down the list until they run out of time.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Rows x cols:", df.shape)
print("Position tiers present:", df['position_tier'].dropna().unique().tolist())

Rows x cols: (30000, 44)
Position tiers present: ['striking', 'page_3_5', 'page_1', 'top_3', 'deep']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `ctr_gap` = `ctr_bench(position_tier) − ctr`**, where `ctr_bench` is the median `ctr` of other pages in the *same* `position_tier`. A positive gap means a page is under-clicking for how well it ranks; the bigger the gap, the higher the priority.

**This is a proxy, and I want to be honest about that.** Both ingredients — `ctr` and `position_tier` (from `avg_position`) — are observed GSC metrics, not someone's manual verdict, so it isn't arbitrary. But it's still a *defined comparison*, not an observed outcome like "this page got rewritten and CTR went up." Benchmarking against the peer median (not one global threshold) is what keeps it honest — "good CTR" for `top_3` and for `deep` are not the same number, and the gap adapts to that. The real test of whether a flagged gap is *worth* a rewrite only comes from an observed after-the-fact outcome (did CTR actually move once the page was touched) — that's warehouse-scale, forward-looking work for a later week, not this slice.

**Leakage guard:** `trend_direction` / `trend_pct` are a different lane's label and must never feed this one — they're excluded from both the target and any future feature set here.

In [4]:
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

vis = df[(df["impressions_90d"] >= 500) & df["position_tier"].notna() & df["ctr"].notna()].copy()
ctr_bench = vis.groupby("position_tier")["ctr"].median().reindex(tier_order)
print("Peer-median CTR by position tier (%):")
print(ctr_bench)

vis["ctr_bench"] = vis["position_tier"].map(ctr_bench)
vis["ctr_gap"] = vis["ctr_bench"] - vis["ctr"]
print("\nEligible pages:", len(vis))
print("Pages flagged as underperforming (gap > 0):", (vis["ctr_gap"] > 0).sum())

Peer-median CTR by position tier (%):
position_tier
top_3       0.20
page_1      0.24
striking    0.17
page_3_5    0.09
deep        0.00
Name: ctr, dtype: float64

Eligible pages: 16726
Pages flagged as underperforming (gap > 0): 7950


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'm using top-K capture lift: take the K pages with the biggest gap and check if their average gap is actually bigger than the average gap among all underperforming pages — if it is, the score's pulling out real outliers and not just noise. Right now, for the top 500 pages, that's 2.32x the average, so the ranking is doing something real. I'm not using precision@K in the classic sense because there's no ground-truth "this page needed a rewrite" label yet — I'd only get that by actually fixing pages and watching if CTR moves, which is later, forward-looking work. This lift number is just what I can defend today, against my current peer-median baseline.

In [5]:
K = 500
topk = vis.sort_values("ctr_gap", ascending=False).head(K)
pop_underperf_mean = vis.loc[vis["ctr_gap"] > 0, "ctr_gap"].mean()
topk_mean = topk["ctr_gap"].mean()

print(f"Mean gap, top-{K} flagged pages:      {topk_mean:.3f} pts CTR")
print(f"Mean gap, all underperforming pages: {pop_underperf_mean:.3f} pts CTR")
print(f"Top-{K} capture lift: {topk_mean / pop_underperf_mean:.2f}x")

Mean gap, top-500 flagged pages:      0.240 pts CTR
Mean gap, all underperforming pages: 0.103 pts CTR
Top-500 capture lift: 2.32x


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), scoped to my lane's eligible slice: `position_tier` known and `impressions_90d ≥ 500` (the volume floor the data dictionary warns is necessary — at low volumes a single click swings CTR by whole points, which would make the gap score noise, not signal).

In [6]:
unit_of_analysis = vis[[
    "content_id", "client_id", "content_type", "main_intent",
    "position_tier", "impressions_90d", "ctr", "ctr_bench", "ctr_gap"
]].sort_values("ctr_gap", ascending=False)

print(unit_of_analysis.shape)
unit_of_analysis.head(10)

(16726, 9)


,content_id,client_id,content_type,main_intent,position_tier,impressions_90d,ctr,ctr_bench,ctr_gap
94,content_9983d31c53cb,client_4e07408562,keyword article,transactional,page_1,7737,0.0,0.24,0.24
11909,content_bb485e06e530,client_19581e27de,keyword article,transactional,page_1,2080,0.0,0.24,0.24
11912,content_cab6d15a5215,client_f74efabef1,keyword article,informational,page_1,2092,0.0,0.24,0.24
29983,content_6880eb215048,client_19581e27de,keyword article,transactional,page_1,2845,0.0,0.24,0.24
29886,content_04d69956e256,client_19581e27de,keyword article,informational,page_1,742,0.0,0.24,0.24
120,content_8f2559c3bc1b,client_6208ef0f77,keyword article,informational,page_1,740,0.0,0.24,0.24
6648,content_10d038b1695e,client_19581e27de,keyword article,informational,page_1,599,0.0,0.24,0.24
1262,content_62032de289e3,client_19581e27de,keyword article,transactional,page_1,622,0.0,0.24,0.24
6559,content_4500346533b1,client_3fdba35f04,keyword article,commercial,page_1,705,0.0,0.24,0.24
6537,content_7caef6b6a306,client_d029fa3a95,comparison article,informational,page_1,6594,0.0,0.24,0.24


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

My peer-median rule already beats one flat cutoff, but it still only looks at one thing: position_tier. "Normal CTR" for a rank actually depends on content_type, main_intent, keyword search volume/competition, freshness — and these interact. I can see it directly: within just the page_1 tier, median CTR goes from 0.00 for comparison articles to 0.26 for feedly articles — same rank, totally different normal. Writing an if-statement for every tier × content_type × intent combo (each needing enough rows to trust) isn't realistic. A model can learn a page's expected CTR from all these fields at once and give one fair score, instead of a rule that ends up over-flagging whole categories just because of how it's built.

In [7]:
page1 = vis[vis["position_tier"] == "page_1"]
print("Median CTR within page_1 tier, split by content_type:")
print(page1.groupby("content_type")["ctr"].median())
print("\nRow counts per group (so the split isn't just small-sample noise):")
print(page1["content_type"].value_counts())

Median CTR within page_1 tier, split by content_type:
content_type
comparison article    0.00
feedly article        0.26
keyword article       0.24
Name: ctr, dtype: float64

Row counts per group (so the split isn't just small-sample noise):
content_type
keyword article       6931
feedly article          83
comparison article      50
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.